### Note :- Im having some issues with Open AI API key - Going ahead with opensource GROQ

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [40]:
from typing import Annotated, TypedDict, List
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
)
import os
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [60]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000169BC777750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000169BC858190>, model_name='llama-3.1-8b-instant', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### 0 - Warm up

In [49]:
messages = [
    HumanMessage("What is the capital of France?")
]


In [51]:
res = llm.invoke(messages)
res.content

'The capital of France is Paris.'

In [52]:
messages2 = [
    SystemMessage("Answer in only one word."),
    HumanMessage("What is the capital of France?")
]

In [53]:
res = llm.invoke(messages2)
res.content

'Paris.'

### 1- A reusable prompt template

In [69]:
prod_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a concise marketing copywriter."),
        ("human", "Write a one line description of {product} for {audience}.")
    ]
)

In [70]:
filled_prompt = prod_prompt.invoke(
    {
        "product" : "wireless earbuds",
        "audience": "college students"
    }
)

In [71]:
for message in filled_prompt.messages:
    print(f"{message.type.upper()}: {message.content}")


SYSTEM: You are a concise marketing copywriter.
HUMAN: Write a one line description of wireless earbuds for college students.


### 2- Turn it into a chain

In [72]:
chain = prod_prompt | llm

In [73]:
# Single invocation
response = chain.invoke(
    {
        "product": "noise-cancelling headphones",
        "audience": "students",
    }
)

print(response.content)

"Immerse yourself in focus with our noise-cancelling headphones, designed to block distractions and help you power through even the longest study sessions."


In [74]:
# Batch invocation
responses = chain.batch(
    [
        {
            "product": "noise-cancelling headphones",
            "audience": "students",
        },
        {
            "product": "smart water bottle",
            "audience": "fitness enthusiasts",
        },
        {
            "product": "portable espresso maker",
            "audience": "travelers",
        },
    ]
)

In [75]:
print("\nBatch results:")
for i, response in enumerate(responses, start=1):
    print(f"{i}. {response.content}")


Batch results:
1. "Immerse yourself in focus with our noise-cancelling headphones, designed to block distractions and help you power through even the longest study sessions."
2. "Hydrate with precision: our smart water bottle tracks your intake, monitors temperature, and sends reminders to stay on top of your fitness goals."
3. "Fuel your adventures on-the-go with the compact, battery-powered EspressoGenie, the ultimate portable espresso maker for travelers."


### 3-Structured output

In [76]:
class Product(BaseModel):
    name: str = Field(description="A catchy product name")
    tagline: str = Field(description="A one-line tagline")
    price_usd: float = Field(description="A reasonable suggested price in USD")

In [77]:
structured_llm_output = llm.with_structured_output(Product)

In [80]:
result = structured_llm_output.invoke("Invent a smart water bottle for gym-goers.")
result

Product(name='HydraFit', tagline='Stay hydrated, stay fit', price_usd=49.99)

In [84]:
result.name

'HydraFit'

In [85]:
result.tagline

'Stay hydrated, stay fit'

In [87]:
result.price_usd

49.99

### Stretch -  a parser at the end

In [103]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()


In [104]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a product designer.
            Generate exactly 5 feature names.

            Rules:
            - Return ONLY the feature names.
            - Separate each feature using a comma.
            - Do not add new lines.
            - Do not add numbering.
            - Do not add explanations.
            - Do not add extra commas."""
        ),
        (
            "human",
            "Product: {product}"
        )
    ]
)

In [105]:
feature_chain = prompt | llm | parser

features = feature_chain.invoke(
    {
        "product": result
    }
)

In [106]:
print(features)

['SmartWaterTracker', 'FitnessGoalSetter', 'HydrationReminder', 'CalorieCounter', 'WorkoutLogger']


In [107]:

print(type(features))

<class 'list'>


### 1. Where did the system message actually change the output?

The system message controls the model's behavior and instructions.It acts as an information extraction assistant and extract only the requested fields.

### 2. Why is structured output safer than parsing the model's plain text yourself?

lets say for an entity age, instead of int value we are giving string then the model will interpret as string and at the end it may leads to wrong response. thats why we use structed output format like pydantic etc.

### 3. What did the | pipe save you compared to calling each piece by hand?

instead of manually connecting each component, the | operator composes them into a single reusable chain, making the code shorter, easier to read, and easier to extend (e.g.:  prompt | llm | parser)